# Silver Enriched: Store Dimension (SCD Type 2)

**Job Name:** `store_dim_scd2`  
**Source:** `bronze.orders_raw` (via CDF)  
**Target:** `silver_enriched.store_dim`  
**SCD2 Trigger:** Change in `store_type`, `region`, `city`, or `state`

## Import Utils

In [ ]:
import sys
sys.path.append("/opt/spark-notebooks/silver_enriched")

from utils import (
    get_last_processed_version,
    save_last_processed_version,
    read_bronze_cdf,
    get_current_bronze_version
)

print("Utils imported ✅")

## Create Table

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver_enriched")

spark.sql("""
    CREATE TABLE IF NOT EXISTS silver_enriched.store_dim (
        store_sk BIGINT,
        store_id STRING,
        store_type STRING,
        region STRING,
        city STRING,
        state STRING,
        effective_from TIMESTAMP,
        effective_to TIMESTAMP,
        is_current BOOLEAN
    )
    USING DELTA
    LOCATION '/opt/spark-data/delta/silver/enriched/store_dim'
""")

print("Table ready ✅")

## Check Last Processed Version

In [ ]:
JOB_NAME = "store_dim_scd2"

last_version = get_last_processed_version(spark, JOB_NAME)
print(f"Last processed version: {last_version}")

## Read Only New Data from Bronze (CDF)

In [ ]:
bronze_df = read_bronze_cdf(spark, last_version)
new_row_count = bronze_df.count()
print(f"New rows to process: {new_row_count}")

## Parse & Deduplicate Store Attributes

In [ ]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window

if new_row_count > 0:
    store_schema = StructType([
        StructField("store_details", StructType([
            StructField("store_id", StringType()),
            StructField("city", StringType()),
            StructField("state", StringType()),
            StructField("region", StringType()),
            StructField("store_attributes", StructType([
                StructField("store_type", StringType())
            ]))
        ])),
        StructField("order", StructType([
            StructField("order_time", StringType())
        ]))
    ])

    parsed = bronze_df.withColumn("data", from_json(col("raw_payload"), store_schema))

    incoming_stores = parsed.select(
        col("data.store_details.store_id").alias("store_id"),
        col("data.store_details.store_attributes.store_type").alias("store_type"),
        col("data.store_details.region").alias("region"),
        col("data.store_details.city").alias("city"),
        col("data.store_details.state").alias("state"),
        col("data.order.order_time").cast("timestamp").alias("order_time")
    )

    # Deduplicate: Keep the LATEST version per store_id
    w = Window.partitionBy("store_id").orderBy(col("order_time").desc())

    latest_stores = (
        incoming_stores
        .withColumn("rn", row_number().over(w))
        .filter("rn = 1")
        .drop("rn", "order_time")
    )

    latest_stores.createOrReplaceTempView("incoming_stores")
    print(f"Unique stores to process: {latest_stores.count()}")
    latest_stores.show(truncate=False)
else:
    print("No new data. Skipping. ✅")

## SCD2 MERGE

In [ ]:
if new_row_count > 0:
    existing_count = spark.sql("SELECT COUNT(*) FROM silver_enriched.store_dim").collect()[0][0]

    if existing_count == 0:
        print("Seeding dimension...")
        spark.sql("""
            INSERT INTO silver_enriched.store_dim
            SELECT
                monotonically_increasing_id() AS store_sk,
                store_id, store_type, region, city, state,
                current_timestamp() AS effective_from,
                CAST('9999-12-31 00:00:00' AS TIMESTAMP) AS effective_to,
                true AS is_current
            FROM incoming_stores
        """)
        print("Dimension seeded ✅")

    else:
        print("Running SCD2 merge...")
        spark.sql("""
            MERGE INTO silver_enriched.store_dim AS target
            USING incoming_stores AS source
            ON target.store_id = source.store_id AND target.is_current = true
            WHEN MATCHED AND (
                target.store_type != source.store_type OR
                target.region     != source.region     OR
                target.city       != source.city       OR
                target.state      != source.state
            )
            THEN UPDATE SET
                target.is_current   = false,
                target.effective_to = current_timestamp()
        """)
        print("Step 1 done: Closed old versions.")

        spark.sql("""
            INSERT INTO silver_enriched.store_dim
            SELECT
                (SELECT COALESCE(MAX(store_sk), 0) FROM silver_enriched.store_dim)
                    + monotonically_increasing_id() + 1 AS store_sk,
                src.store_id, src.store_type, src.region, src.city, src.state,
                current_timestamp() AS effective_from,
                CAST('9999-12-31 00:00:00' AS TIMESTAMP) AS effective_to,
                true AS is_current
            FROM incoming_stores src
            LEFT ANTI JOIN silver_enriched.store_dim dim
                ON src.store_id = dim.store_id AND dim.is_current = true
        """)
        print("Step 2 done: Inserted new versions ✅")

    current_version = get_current_bronze_version(spark)
    save_last_processed_version(spark, JOB_NAME, current_version, new_row_count)

## Verify

In [ ]:
spark.sql("""
    SELECT store_sk, store_id, store_type, region, city, state,
           is_current, effective_from, effective_to
    FROM silver_enriched.store_dim
    ORDER BY store_id, effective_from
""").toPandas()